# Tiền xử lý dữ liệu cho các mô hình RoBERTa (NLBSE'23 Code Comment Classification)

## Mục tiêu
Notebook này chỉ thực hiện **tiền xử lý dữ liệu thuần túy** — không tokenize, không xây dựng model, không tính class weights.
Tất cả các bước liên quan đến training (tokenize, DataLoader, loss weights…) được thực hiện riêng trong từng training notebook.

## Output: 6 file chuẩn
```
data/roberta/
├── train_meta.csv    # comment_sentence_id | text | label_cols — dùng cho training
├── test_meta.csv     # comment_sentence_id | text | label_cols — chỉ dùng để đánh giá cuối
├── y_train.npy       # (N_train, 5) float32 — multi-label ground truth
├── y_test.npy        # (N_test,  5) float32 — multi-label ground truth
├── folds.json        # 5-fold CV indices trên tập Train
└── label_cols.json   # thứ tự tên các label
```

## Quy trình
- **Bước 0**: Cài đặt / kiểm tra các thư viện cần thiết.
- **Bước 1**: Nạp dữ liệu và kiểm tra schema.
- **Bước 2**: Chuẩn hóa văn bản comment.
- **Bước 3**: Phân chia Train/Test (80/20) bằng `MultilabelStratifiedKFold`.
- **Bước 4**: Tạo 5-fold Cross-Validation trên tập Train.
- **Bước 5**: Lưu 6 file output.
- **Bước 6**: Kiểm tra xác thực.

In [ ]:
# Bước 0: Kiểm tra / cài đặt thư viện
import subprocess, sys

required = ['iterstrat']
missing  = []
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'Đang cài đặt: {missing}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'iterative-stratification'])
    print('Cài đặt xong!')
else:
    print('Tất cả thư viện đã có sẵn!')

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# ── Cấu hình đường dẫn ────────────────────────────────────────────────────────
DATA_DIR   = os.path.abspath('../data')
OUTPUT_DIR = os.path.join(DATA_DIR, 'roberta')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Cấu hình chung ────────────────────────────────────────────────────────────
LABEL_COLS   = ['DevelopmentNotes', 'Expand', 'Parameters', 'Summary', 'Usage']
N_FOLDS      = 5
RANDOM_STATE = 42

print(f'Thư mục dữ liệu  : {DATA_DIR}')
print(f'Thư mục output   : {OUTPUT_DIR}')
print(f'Labels           : {LABEL_COLS}')
print(f'CV folds         : {N_FOLDS}')

## Bước 1: Nạp dữ liệu và kiểm tra schema

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'python_preprocessed.csv'))

print(f'Tổng số mẫu   : {len(df):,}')
print(f'Số cột        : {df.shape[1]}')
print(f'Cột dữ liệu   : {df.columns.tolist()}')
print(f'Missing values: {df.isnull().sum().sum()}')
print()

print('Thống kê positive per label:')
stats = df[LABEL_COLS].sum().to_frame('Count')
stats['Ratio (%)'] = (stats['Count'] / len(df) * 100).round(2)
print(stats)

## Bước 2: Chuẩn hóa văn bản comment
Với RoBERTa **không cần lowercase** hay loại bỏ dấu câu vì tokenizer BPE của RoBERTa đã xử lý.
Chỉ chuẩn hóa khoảng trắng và ký tự điều khiển — kết quả lưu vào cột `text` của meta CSV.

In [ ]:
def normalize_text(text: str) -> str:
    """Chuẩn hóa khoảng trắng và ký tự điều khiển."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['comment_sentence'].apply(normalize_text)

# Thống kê độ dài (word count) — chỉ để tham khảo khi chọn MAX_LEN trong training
df['word_count'] = df['text'].str.split().str.len()
print('Thống kê độ dài câu (word count):')
print(df['word_count'].describe().round(2))
pct95 = int(df['word_count'].quantile(0.95))
print(f'\n95th percentile: {pct95} words  (tham khảo khi chọn MAX_LEN trong training notebook)')

## Bước 3: Phân chia Train / Test (80/20) bằng Multilabel Stratification
`MultilabelStratifiedKFold` đảm bảo tỷ lệ phân bố của cả 5 nhãn cân bằng giữa Train và Test.

> **Lưu ý**: `test_meta.csv` và `y_test.npy` **chỉ dùng để đánh giá cuối cùng**, không tham gia
> huấn luyện hay chọn hyperparameter.

In [ ]:
mskf_split = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
Y_all      = df[LABEL_COLS].values

# Dùng fold 0 làm Test (≈20%), phần còn lại là Train (≈80%)
train_idx, test_idx = next(mskf_split.split(np.zeros(len(df)), Y_all))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

print(f'Tập Train: {len(train_df):,} mẫu ({len(train_df)/len(df)*100:.1f}%)')
print(f'Tập Test : {len(test_df):,}  mẫu ({len(test_df)/len(df)*100:.1f}%)')

# Kiểm tra không bị rò rỉ dữ liệu
overlap = set(train_df['comment_sentence_id']) & set(test_df['comment_sentence_id'])
assert len(overlap) == 0, f'Data leakage: {len(overlap)} IDs trùng!'
print('✅ Không có data leakage')

# So sánh phân bố nhãn
print('\nPhân bố nhãn Train vs Test:')
comp = pd.DataFrame({
    'Train (%)': (train_df[LABEL_COLS].mean() * 100).round(2),
    'Test  (%)': (test_df[LABEL_COLS].mean()  * 100).round(2),
})
print(comp)

## Bước 4: Tạo 5-Fold Cross-Validation trên tập Train
`folds.json` cho phép **tất cả 9 mô hình** dùng chung một cách chia dữ liệu → đảm bảo so sánh công bằng.
Mỗi entry lưu `train_indices` và `val_indices` là index trong `train_meta.csv`.

In [ ]:
mskf_cv = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
folds   = []

for fold_idx, (trn_i, val_i) in enumerate(
        mskf_cv.split(np.zeros(len(train_df)), train_df[LABEL_COLS].values)):
    folds.append({
        'fold':          fold_idx,
        'train_indices': trn_i.tolist(),
        'val_indices':   val_i.tolist(),
    })
    print(f'Fold {fold_idx}: Train={len(trn_i):,}  Val={len(val_i):,}')

print(f'\nTổng {N_FOLDS} folds đã tạo xong.')

## Bước 5: Lưu 6 file output

In [ ]:
# ── 1. train_meta.csv ─────────────────────────────────────────────────────────
# Lưu id, raw comment và các cột metadata (bao gồm label để tiện tham chiếu)
meta_cols = ['comment_sentence_id', 'text'] + LABEL_COLS
train_df[meta_cols].to_csv(os.path.join(OUTPUT_DIR, 'train_meta.csv'), index=False)

# ── 2. test_meta.csv ──────────────────────────────────────────────────────────
test_df[meta_cols].to_csv(os.path.join(OUTPUT_DIR, 'test_meta.csv'), index=False)

# ── 3 & 4. y_train.npy / y_test.npy ──────────────────────────────────────────
y_train = train_df[LABEL_COLS].values.astype(np.float32)
y_test  = test_df[LABEL_COLS].values.astype(np.float32)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'),  y_test)

# ── 5. folds.json ─────────────────────────────────────────────────────────────
folds_path = os.path.join(OUTPUT_DIR, 'folds.json')
with open(folds_path, 'w', encoding='utf-8') as f:
    json.dump(folds, f, indent=2)

# ── 6. label_cols.json ────────────────────────────────────────────────────────
label_path = os.path.join(OUTPUT_DIR, 'label_cols.json')
with open(label_path, 'w', encoding='utf-8') as f:
    json.dump(LABEL_COLS, f, indent=2)

# ── Tóm tắt kích thước file ───────────────────────────────────────────────────
OUTPUT_FILES = [
    'train_meta.csv', 'test_meta.csv',
    'y_train.npy',    'y_test.npy',
    'folds.json',     'label_cols.json',
]

print(f'Artifacts đã lưu vào: {OUTPUT_DIR}')
print(f'{"File":<22}  {"Shape / Info":<25}  Size')
print('-' * 62)
for fname in OUTPUT_FILES:
    fpath   = os.path.join(OUTPUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1024**2
    if fname.endswith('.npy'):
        arr   = np.load(fpath)
        extra = f'{arr.shape}  dtype={arr.dtype}'
    elif fname.endswith('.csv'):
        tmp   = pd.read_csv(fpath)
        extra = f'{len(tmp):,} rows × {tmp.shape[1]} cols'
    elif fname == 'folds.json':
        with open(fpath) as jf:
            extra = f'{len(json.load(jf))} folds'
    else:
        with open(fpath) as jf:
            extra = str(json.load(jf))
    print(f'  {fname:<22}  {extra:<25}  {size_mb:.3f} MB')

## Bước 6: Kiểm tra xác thực (Sanity Check)

In [ ]:
# Tải lại từ disk để xác nhận tính nhất quán
tr_meta = pd.read_csv(os.path.join(OUTPUT_DIR, 'train_meta.csv'))
te_meta = pd.read_csv(os.path.join(OUTPUT_DIR, 'test_meta.csv'))
y_tr    = np.load(os.path.join(OUTPUT_DIR, 'y_train.npy'))
y_te    = np.load(os.path.join(OUTPUT_DIR, 'y_test.npy'))
with open(os.path.join(OUTPUT_DIR, 'folds.json'))      as f: folds_loaded  = json.load(f)
with open(os.path.join(OUTPUT_DIR, 'label_cols.json')) as f: labels_loaded = json.load(f)

# Kiểm tra shape nhất quán
assert len(tr_meta)    == y_tr.shape[0],       'train_meta và y_train không khớp số dòng!'
assert len(te_meta)    == y_te.shape[0],       'test_meta và y_test không khớp số dòng!'
assert y_tr.shape[1]   == len(labels_loaded),  'y_train số cột ≠ số label!'
assert y_te.shape[1]   == len(labels_loaded),  'y_test số cột ≠ số label!'
assert len(folds_loaded) == N_FOLDS,           f'folds.json có {len(folds_loaded)} fold ≠ {N_FOLDS}!'

# Kiểm tra không leakage giữa train và test
overlap = set(tr_meta['comment_sentence_id']) & set(te_meta['comment_sentence_id'])
assert len(overlap) == 0, f'Data leakage: {len(overlap)} IDs trùng!'

# Kiểm tra tất cả fold indices nằm trong phạm vi train
n_train = len(tr_meta)
for fd in folds_loaded:
    all_idx = fd['train_indices'] + fd['val_indices']
    assert max(all_idx) < n_train and min(all_idx) >= 0, f"Fold {fd['fold']}: index out of range!"
    assert set(fd['train_indices']).isdisjoint(fd['val_indices']), f"Fold {fd['fold']}: train/val overlap!"

# In tóm tắt
val_sizes = ', '.join(str(len(fd['val_indices'])) for fd in folds_loaded)
print('=' * 60)
print('  DATA PREPROCESSING SUMMARY')
print('=' * 60)
print(f'  train_meta.csv   : {len(tr_meta):,} rows  →  y_train {y_tr.shape}')
print(f'  test_meta.csv    : {len(te_meta):,}  rows  →  y_test  {y_te.shape}')
print(f'  label_cols.json  : {labels_loaded}')
print(f'  folds.json       : {N_FOLDS} folds  (val sizes: {val_sizes})')
print('=' * 60)
print()
print('✅ Tất cả 6 file output hợp lệ — sẵn sàng để các training notebook sử dụng!')